# Pareidolia GPU morphology fine-tuning

This notebook fine-tunes complete EfficientNet-B0 and ResNet-18 networks. It screens one grouped fold, then runs three grouped folds only for the best architecture/policy. It never replaces the fallback unless the prespecified morphology gate passes.

In [ ]:
import os, subprocess, sys, torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU, then rerun.'
print(torch.__version__, torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
from pathlib import Path
repo = Path('/content/pareidolia-paradox')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'codex/competition-delivery', 'https://github.com/Aniket-Nikam/pareidolia-paradox-competition-2026.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown', 'scikit-learn==1.9.0', 'Pillow==12.1.1'], check=True)

## Dataset
The next cell downloads the organizer's public Drive folder. If Drive throttles the folder, upload the two original outer ZIP files into `/content/organizer_download` and rerun the preparation cell.

In [ ]:
download = Path('/content/organizer_download')
download.mkdir(exist_ok=True)
if not list(download.rglob('*.zip')):
    subprocess.run(['gdown', '--folder', 'https://drive.google.com/drive/folders/1nSdNXvR0ZMMqPoxn-iijvV5gBZLIXd5p?usp=sharing', '-O', str(download)], check=True)
print([str(path) for path in download.rglob('*') if path.is_file()])

In [ ]:
data = Path('/content/pareidolia_data')
if not data.exists():
    sources = [str(path) for path in download.rglob('*') if path.is_file()]
    subprocess.run([sys.executable, 'prepare_data.py', '--sources', *sources, '--destination', str(data)], check=True)
audit = Path('/content/audit')
if not (audit / 'duplicate_groups.csv').exists():
    subprocess.run([sys.executable, 'data_checks.py', '--data-root', str(data), '--artifacts', str(audit)], check=True)
print('data and duplicate groups ready')

## Upload the preserved fallback CSV
Upload only the existing validated `submission.csv`. It remains untouched and will be selected automatically if the morphology gate fails.

In [ ]:
from google.colab import files
fallback = Path('/content/fallback_submission.csv')
if not fallback.exists():
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one existing submission.csv')
    fallback.write_bytes(next(iter(uploaded.values())))
subprocess.run([sys.executable, 'validate_submission.py', '--submission', str(fallback), '--metadata', str(next(data.rglob('test_metadata.csv')))], check=True)

## Train
The default omits ConvNeXt-Tiny to protect time. Add `--include-convnext` only if GPU memory and deadline permit. Outputs are resumable at the fold-directory level.

In [ ]:
results = Path('/content/gpu_results')
command = [sys.executable, '-m', 'gpu.workflow', '--data-root', str(data), '--groups-csv', str(audit / 'duplicate_groups.csv'), '--reference-metrics', 'reports/reliability_metrics.json', '--fallback-submission', str(fallback), '--output-dir', str(results), '--screen-epochs', '6', '--full-epochs', '18', '--patience', '4', '--batch-size', '64', '--workers', '2']
subprocess.run(command, check=True)

In [ ]:
print((results / 'updated_reliability_report.md').read_text())
print((results / 'best_verified' / 'final_recommendation.json').read_text())

In [ ]:
import shutil
archive = shutil.make_archive('/content/pareidolia_gpu_results', 'zip', results)
files.download(str(results / 'best_verified' / 'submission.csv'))
files.download(archive)